In [4]:
import pandas as pd
import numpy as np

# Carga
df = pd.read_csv("../../data/raw/USW00014607.csv", low_memory=False)

# Guardado de datos limpios
df.to_csv("../../data/interim/weather_cleaned.csv", index=False)

# Guardado de datos procesados
df.to_csv("../../data/processed/weather_processed.csv", index=False)

# Parseo de fecha
df["DATE"] = pd.to_datetime(df["DATE"])

# Filtro temporal
df = df[(df["DATE"] >= "2020-01-01") & (df["DATE"] <= "2025-12-31")]

df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(2192, 144)


,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,WT19,WT19_ATTRIBUTES,WT21,WT21_ATTRIBUTES,WT22,WT22_ATTRIBUTES,WV03,WV03_ATTRIBUTES,WV20,WV20_ATTRIBUTES
0,USW00014607,2020-01-01,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",10.0,",,W,2400",13.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00014607,2020-01-02,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",0.0,"T,,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USW00014607,2020-01-03,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",3.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USW00014607,2020-01-04,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",0.0,"T,,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USW00014607,2020-01-05,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",13.0,",,W,2400",18.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Target: lluvia (1 si llueve, 0 si no)
df["Rain"] = (df["PRCP"] > 0).astype(int)

# Eliminamos PRCP si queremos evitar leakage directo
# (esto lo vamos a comparar después, pero por ahora lo sacamos)
df = df.drop(columns=["PRCP"])

In [6]:
# Eliminar columnas tipo attributes
attr_cols = [col for col in df.columns if "ATTRIBUTES" in col]
df = df.drop(columns=attr_cols)

# Eliminar columnas de flags muy ruidosas (evaluamos luego si sumar alguna)
wt_cols = [col for col in df.columns if col.startswith("WT")]
wv_cols = [col for col in df.columns if col.startswith("WV")]

df = df.drop(columns=wt_cols + wv_cols)

In [7]:
# % de NaNs
missing_ratio = df.isna().mean()

# Eliminar columnas con muchos NaNs (>50%)
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
df = df.drop(columns=cols_to_drop)

print(f"Columnas eliminadas por NaN: {len(cols_to_drop)}")

Columnas eliminadas por NaN: 29


In [8]:
# Columnas sin variación
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
df = df.drop(columns=constant_cols)

print(f"Columnas constantes eliminadas: {len(constant_cols)}") 

Columnas constantes eliminadas: 5


In [ ]:
df["year"] = df["DATE"].dt.year
df["month"] = df["DATE"].dt.month
df["day"] = df["DATE"].dt.day
df["dayofweek"] = df["DATE"].dt.dayofweek

# Opcional: estacionalidad (mejor que mes crudo)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Eliminamos DATE (no usable directo)
df = df.drop(columns=["DATE"])

In [13]:
df.to_csv("../../data/interim/weather_cleaned.csv", index=False)

In [14]:
X = df.drop(columns=["Rain"])
y = df["Rain"]

print(X.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

(2192, 22)


## Dataset Summary

- Periodo: 2020-2025
- Filas: 2192
- Variables originales: ~144
- Variables finales: 22

Se realizó:
- limpieza de columnas irrelevantes
- manejo de valores faltantes
- eliminación de variables constantes
- feature engineering temporal

In [15]:
df.to_csv("../../data/processed/weather_processed.csv", index=False)